# 4.4 — Comparaison OpenAI / Mistral sur toute la partie QA

On refait **tous les appels au LLM** de la partie QA avec un second fournisseur, Mistral, pour voir si les conclusions
dépendent du modèle choisi.

- **Mêmes prompts, mot pour mot** : chaque prompt envoyé à OpenAI est relu dans les fichiers de prédictions et renvoyé
  tel quel à Mistral (mêmes passages, même ordre, mêmes consignes). Seul le modèle change.
- **Modèle** : `mistral-medium-2604` (version datée, même gamme que `gpt-4.1-mini` : modèle intermédiaire, sans
  raisonnement caché), température 0, graine 42 (`random_seed` chez Mistral), 300 tokens au plus. Clé
  `MISTRAL_API_KEY` dans `.env` ; cache séparé `resultats/qa/cache_llm_mistral.json`.
- **Expériences reprises** : DEV (Q1/Q2, prompts v1 et v2, contextes c1 et c2), Q2 avec la recherche dense, essai
  3 / 5 / 10 passages, TEST (configuration figée : Q1, Q2, Q2 dense) et la question libre sur la chimiothérapie
  adjuvante / néoadjuvante. Q0 n'utilise pas de LLM : il est identique pour les deux.
- **TEST** : la configuration figée n'est pas modifiée et rien n'est réglé à partir de ces résultats ; on compare
  seulement deux LLM. La grille manuelle (B.5) n'est pas refaite : on compare les contrôles automatiques et BLEU/ROUGE,
  puis on lit les réponses.

In [1]:
import sys, json, statistics
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.qa_llm import ClientLLM, formater_passages, diagnostiquer
from src.qa_recherche import construire_bm25, scores_bm25, premiers
from src.qa_scores import scores_reponses

DATA, RES = RACINE / "data", RACINE / "resultats" / "qa"
gel = json.loads((RES / "qa_config_gelee.json").read_text(encoding="utf-8"))
p = gel["llm"]
openai_llm = ClientLLM(p["modele"], p["temperature"], p["seed"], p["max_tokens"], RES / "cache_llm.json")
MODELE_MISTRAL = "mistral-medium-2604"
mistral_llm = ClientLLM(MODELE_MISTRAL, p["temperature"], p["seed"], p["max_tokens"], RES / "cache_llm_mistral.json",
                        fournisseur="mistral")

def lire(chemin):
    return [json.loads(l) for l in Path(chemin).read_text(encoding="utf-8").splitlines()]

questions = {q["question_id"]: q for f in ("questions_dev.jsonl", "questions_test.jsonl") for q in lire(DATA / f)}
passages = {x["passage_id"]: x for x in lire(DATA / "passages.jsonl")}
catalogue = pd.read_csv(DATA / "corpus.csv", dtype=str)
TITRES = dict(zip(catalogue.document_id, catalogue.titre.str.replace(
    r" \((page web|guide patients[^)]*|fiche Cancer info)\)$", "", regex=True)))   # comme dans le notebook 4.2

## Les réponses OpenAI déjà produites

Une « expérience » = un ensemble de questions × une configuration. Les réponses OpenAI sont relues dans les fichiers
des notebooks 4.2 et 5 ; la question libre est reconstruite comme dans le notebook 4.2 (ses réponses OpenAI viennent du
cache).

In [2]:
SOURCES = {   # fichier de prédictions OpenAI → nom de l'expérience de chaque ligne
    RES / "predictions_llm_dev.jsonl": lambda l: f"DEV {l['configuration']} {l['prompt_version'][3:]} {l['contexte_version']}",
    RES / "predictions_llm_dev_dense.jsonl": lambda l: "DEV Q2 v1 c2, dense",
    RES / "predictions_llm_dev_nb_passages.jsonl": lambda l: f"DEV Q2 v1 c2, k = {l['nb_passages']}",
    DATA / "predictions_qa.jsonl": lambda l: f"TEST {l['configuration']} v1 c2",
    RES / "predictions_llm_test_dense.jsonl": lambda l: "TEST Q2 v1 c2, dense",
}
CHAMPS = ["experience", "configuration", "question_id", "partition", "prompt_version", "contexte_version", "recherche",
          "passages_retrouves", "passages_transmis", "prompt"]
openai_lignes, q0_lignes = [], []
for fichier, nom in SOURCES.items():
    for l in lire(fichier):
        if l["configuration"] == "Q0":
            q0_lignes.append({**l, "experience": "TEST Q0"})
        else:
            openai_lignes.append({**l, "experience": nom(l)})
q0_lignes += [{**l, "experience": "DEV Q0"} for l in lire(RES / "predictions_q0_dev.jsonl")]

# Question libre (notebook 4.2) : 5 passages BM25 REC-v2, prompts figés Q1 et Q2, contexte c2.
QUESTION_LIBRE = "Dans le cancer du sein, dans quel cas doit-on avoir une chimio adjuvante, ou néoadjuvante ?"
ordre = list(passages)
bm25 = construire_bm25([passages[i]["texte"] for i in ordre], racines=True)
top = [passages[ordre[i]] for i, _ in premiers(scores_bm25(QUESTION_LIBRE, bm25), gel["recherche"]["k"])]
for c in ("Q1", "Q2"):
    prompt = gel["configurations"][c]["prompt"].format(question=QUESTION_LIBRE, passages=formater_passages(top, TITRES))
    s = openai_llm.generer(prompt)
    assert s["origine"] == "cache"
    openai_lignes.append({"experience": f"LIBRE {c} v1 c2", "configuration": c, "question_id": "LIBRE", "partition": "libre",
                          "prompt_version": gel["configurations"][c]["prompt_version"], "contexte_version": "c2",
                          "recherche": gel["recherche"]["socle"], "passages_retrouves": [{"passage_id": x["passage_id"]} for x in top],
                          "passages_transmis": [x["passage_id"] for x in top], "prompt": prompt, "reponse": s["texte"],
                          **{k: s.get(k) for k in ("tokens_prompt", "tokens_reponse", "duree_secondes", "fin")}})
print(len(openai_lignes), "réponses OpenAI à reproduire ;", pd.Series([l["experience"] for l in openai_lignes]).value_counts(sort=False).to_dict())

192 réponses OpenAI à reproduire ; {'DEV Q1 v1 c1': 20, 'DEV Q2 v1 c1': 20, 'DEV Q1 v1 c2': 20, 'DEV Q2 v1 c2': 20, 'DEV Q2 v2 c2': 20, 'DEV Q2 v1 c2, dense': 20, 'DEV Q2 v1 c2, k = 3': 20, 'DEV Q2 v1 c2, k = 10': 20, 'TEST Q1 v1 c2': 10, 'TEST Q2 v1 c2': 10, 'TEST Q2 v1 c2, dense': 10, 'LIBRE Q1 v1 c2': 1, 'LIBRE Q2 v1 c2': 1}


## Génération avec Mistral

Une requête par réponse OpenAI, avec le même prompt. En cas d'échec d'un appel, rien n'est écrit (comme au notebook 5).
Sortie : `resultats/qa/predictions_mistral.jsonl`.

In [3]:
mistral_lignes = []
for l in openai_lignes:
    s = mistral_llm.generer(l["prompt"])
    mistral_lignes.append({**{k: l[k] for k in CHAMPS}, "reponse": s["texte"], "echec": s["echec"], "modele": MODELE_MISTRAL,
                           "parametres": {"temperature": p["temperature"], "random_seed": p["seed"], "max_tokens": p["max_tokens"]},
                           **{k: s.get(k) for k in ("modele_reponse", "fin", "tokens_prompt", "tokens_reponse", "duree_secondes", "origine")}})
echecs = [(l["experience"], l["question_id"], l["echec"]) for l in mistral_lignes if l["echec"]]
if echecs:
    raise RuntimeError(f"{len(echecs)} appels en échec, rien n'est écrit ; premier : {echecs[0]}")
with open(RES / "predictions_mistral.jsonl", "w", encoding="utf-8") as f:
    for l in mistral_lignes:
        f.write(json.dumps(l, ensure_ascii=False) + "\n")
print(len(mistral_lignes), "réponses ; origine", pd.Series([l["origine"] for l in mistral_lignes]).value_counts().to_dict(),
      "; réponses tronquées (300 tokens) :", sum(l["fin"] == "length" for l in mistral_lignes))

192 réponses ; origine {'cache': 192} ; réponses tronquées (300 tokens) : 0


## Comparaison chiffrée

Pour chaque expérience et chaque fournisseur : contrôles automatiques (abstentions, citations, longueur) et BLEU / ROUGE
(questions avec réponse, conventions du notebook 4.2, `src/qa_scores.py`). Rappel : ces scores mesurent la ressemblance
de mots avec la référence, pas la justesse.

In [4]:
def resumer(lignes, fournisseur):
    lignes = [l for l in lignes if l["partition"] != "libre"]
    d = pd.DataFrame([{"avec_reponse": questions[l["question_id"]]["answerable"],
                       **diagnostiquer(l["reponse"], l["passages_transmis"])} for l in lignes])
    llm = lignes[0]["configuration"] != "Q0"
    return {"fournisseur": fournisseur,
            "abstentions correctes": f"{int((d.abstention & ~d.avec_reponse).sum())}/{int((~d.avec_reponse).sum())}" if llm else "-",
            "abstentions à tort": int((d.abstention & d.avec_reponse).sum()) if llm else "-",
            "sans citation": int(((d.citations == 0) & ~d.abstention).sum()),
            "citations inventées": int(d.citations_inventees.map(len).sum()),
            **{k: v for k, v in scores_reponses(lignes, questions).items() if k not in ("signature BLEU", "questions")},
            "tokens réponse (médiane)": statistics.median(l.get("tokens_reponse") or 0 for l in lignes) if llm else "-",
            "durée s (médiane)": round(statistics.median(l.get("duree_secondes") or 0 for l in lignes), 2) if llm else "-"}

def tableau(partition):
    noms = list(dict.fromkeys(l["experience"] for l in q0_lignes + openai_lignes if l["partition"] == partition))
    lignes = []
    for nom in noms:
        if nom.endswith("Q0"):
            lignes.append({"expérience": nom, **resumer([l for l in q0_lignes if l["experience"] == nom], "sans LLM")})
            continue
        for fournisseur, toutes in (("OpenAI", openai_lignes), ("Mistral", mistral_lignes)):
            lignes.append({"expérience": nom, **resumer([l for l in toutes if l["experience"] == nom], fournisseur)})
    return pd.DataFrame(lignes).set_index(["expérience", "fournisseur"])

comparaison_dev, comparaison_test = tableau("dev"), tableau("test")
pd.concat({"DEV": comparaison_dev, "TEST": comparaison_test}).to_csv(RES / "comparaison_openai_mistral.csv")
print("DEV : 20 questions (16 avec réponse, 4 sans) ; BLEU/ROUGE sur les 16 avec réponse")
comparaison_dev

DEV : 20 questions (16 avec réponse, 4 sans) ; BLEU/ROUGE sur les 16 avec réponse


abstentions correctes abstentions à tort  \
expérience           fournisseur                                            
DEV Q0               sans LLM                        -                  -   
DEV Q1 v1 c1         OpenAI                        0/4                  0   
                     Mistral                       0/4                  0   
DEV Q2 v1 c1         OpenAI                        4/4                  2   
                     Mistral                       4/4                  1   
DEV Q1 v1 c2         OpenAI                        0/4                  0   
                     Mistral                       0/4                  0   
DEV Q2 v1 c2         OpenAI                        4/4                  2   
                     Mistral                       4/4                  1   
DEV Q2 v2 c2         OpenAI                        4/4                  0   
                     Mistral                       4/4                  1   
DEV Q2 v1 c2, dense  OpenAI                        4/4                  1   
                     Mistral                       4/4                  0   
DEV Q2 v1 c2, k = 3  OpenAI                        4/4                  3   
                     Mistral                       4/4                  4   
DEV Q2 v1 c2, k = 10 OpenAI                        4/4                  2   
                     Mistral                       4/4                  1   

                                  sans citation  citations inventées   BLEU  \
expérience           fournisseur                                              
DEV Q0               sans LLM                 0                    0   5.67   
DEV Q1 v1 c1         OpenAI                   1                    0  13.52   
                     Mistral                  1                    0  20.20   
DEV Q2 v1 c1         OpenAI                   0                    0  16.23   
                     Mistral                  0                    0  17.09   
DEV Q1 v1 c2         OpenAI                   1                    0  13.04   
                     Mistral                  0                    0  19.07   
DEV Q2 v1 c2         OpenAI                   0                    0  16.16   
                     Mistral                  0                    0  20.56   
DEV Q2 v2 c2         OpenAI                   0                    0  13.85   
                     Mistral                  0                    0  20.06   
DEV Q2 v1 c2, dense  OpenAI                   0                    0  14.12   
                     Mistral                  0                    0  16.47   
DEV Q2 v1 c2, k = 3  OpenAI                   0                    0  16.71   
                     Mistral                  0                    0  14.83   
DEV Q2 v1 c2, k = 10 OpenAI                   0                    0  14.28   
                     Mistral                  0                    0  18.86   

                                  ROUGE-1  ROUGE-2  ROUGE-L  réponses vides  \
expérience           fournisseur                                              
DEV Q0               sans LLM       0.276    0.110    0.184               0   
DEV Q1 v1 c1         OpenAI         0.425    0.202    0.336               0   
                     Mistral        0.469    0.264    0.390               0   
DEV Q2 v1 c1         OpenAI         0.448    0.238    0.358               1   
                     Mistral        0.451    0.239    0.345               1   
DEV Q1 v1 c2         OpenAI         0.434    0.209    0.336               0   
                     Mistral        0.485    0.259    0.394               0   
DEV Q2 v1 c2         OpenAI         0.453    0.231    0.360               1   
                     Mistral        0.472    0.262    0.374               1   
DEV Q2 v2 c2         OpenAI         0.445    0.235    0.344               0   
                     Mistral        0.466    0.253    0.372               1   
DEV Q2 v1 c2, dense  OpenAI         0.50

In [5]:
print("TEST : 10 questions (8 avec réponse, 2 sans), configuration figée")
comparaison_test

TEST : 10 questions (8 avec réponse, 2 sans), configuration figée


abstentions correctes abstentions à tort  \
expérience           fournisseur                                            
TEST Q0              sans LLM                        -                  -   
TEST Q1 v1 c2        OpenAI                        0/2                  0   
                     Mistral                       0/2                  0   
TEST Q2 v1 c2        OpenAI                        1/2                  0   
                     Mistral                       2/2                  0   
TEST Q2 v1 c2, dense OpenAI                        2/2                  1   
                     Mistral                       2/2                  1   

                                  sans citation  citations inventées   BLEU  \
expérience           fournisseur                                              
TEST Q0              sans LLM                 0                    0   5.44   
TEST Q1 v1 c2        OpenAI                   0                    0   9.96   
                     Mistral                  0                    0  11.61   
TEST Q2 v1 c2        OpenAI                   0                    0  10.10   
                     Mistral                  0                    0  11.66   
TEST Q2 v1 c2, dense OpenAI                   0                    0  13.99   
                     Mistral                  0                    0  15.34   

                                  ROUGE-1  ROUGE-2  ROUGE-L  réponses vides  \
expérience           fournisseur                                              
TEST Q0              sans LLM       0.297    0.102    0.210               0   
TEST Q1 v1 c2        OpenAI         0.422    0.173    0.315               0   
                     Mistral        0.451    0.193    0.343               0   
TEST Q2 v1 c2        OpenAI         0.430    0.178    0.325               0   
                     Mistral        0.463    0.191    0.359               0   
TEST Q2 v1 c2, dense OpenAI         0.455    0.222    0.354               0   
                     Mistral        0.477    0.225    0.380               0   

                                  mots (médiane) tokens réponse (médiane)  \
expérience           fournisseur                                            
TEST Q0              sans LLM              105.5                        -   
TEST Q1 v1 c2        OpenAI                 55.0                     85.0   
                     Mistral                48.5                     94.0   
TEST Q2 v1 c2        OpenAI                 59.0                     85.0   
                     Mistral                39.5                     80.5   
TEST Q2 v1 c2, dense OpenAI                 63.0                     84.0   
                     Mistral                45.0                     83.5   

                                 durée s (médiane)  
expérience           fournisseur                    
TEST Q0              sans LLM                    -  
TEST Q1 v1 c2        OpenAI                   1.16  
                     Mistral                  1.02  
TEST Q2 v1 c2        OpenAI                   1.14  
                     Mistral                  0.91  
TEST Q2 v1 c2, dense OpenAI                   1.08  
                     Mistral                   1.0

## Les réponses côte à côte

Questions où les deux LLM n'ont pas le même comportement : l'un s'abstient et pas l'autre, ou le nombre de citations
diffère beaucoup. On lit aussi toutes les réponses TEST (configuration figée) et la question libre.

In [6]:
pd.set_option("display.max_colwidth", None)
cle = lambda l: (l["experience"], l["question_id"])
mistral_par = {cle(l): l for l in mistral_lignes}
duo = pd.DataFrame([{"expérience": l["experience"], "question_id": l["question_id"],
                     "abstention OpenAI": diagnostiquer(l["reponse"], l["passages_transmis"])["abstention"],
                     "abstention Mistral": diagnostiquer(mistral_par[cle(l)]["reponse"], l["passages_transmis"])["abstention"],
                     "OpenAI": l["reponse"], "Mistral": mistral_par[cle(l)]["reponse"]} for l in openai_lignes])
# Abstentions différentes, sur les configurations figées (DEV Q1/Q2 v1 c2, TEST, dense)
figees = duo.expérience.str.contains("v1 c2") & ~duo.expérience.str.contains("k =")
duo[figees & (duo["abstention OpenAI"] != duo["abstention Mistral"])].set_index(["expérience", "question_id"])

,,abstention OpenAI,abstention Mistral,OpenAI,Mistral
expérience,question_id,,,,
DEV Q2 v1 c2,Q014,True,False,Des douleurs aiguës peuvent apparaître après deux ou trois semaines de chimiothérapie et disparaissent lorsque la chimiothérapie est finie [DOC005_P003]. Information insuffisante dans les documents fournis pour savoir si des douleurs persistent après la fin du traitement.,Les douleurs aiguës peuvent apparaître après deux ou trois semaines de chimiothérapie et disparaissent une fois le traitement fini. [DOC005_P003]
"DEV Q2 v1 c2, dense",Q014,True,False,Des douleurs aiguës peuvent apparaître après deux ou trois semaines de chimiothérapie et disparaissent lorsque la chimiothérapie est finie [DOC005_P003]. Information insuffisante dans les documents fournis concernant la persistance des douleurs après la fin du traitement.,Des douleurs aiguës peuvent apparaître après deux ou trois semaines de chimiothérapie et disparaissent lorsque le traitement est fini. [DOC005_P003]
TEST Q2 v1 c2,Q030,False,True,"Les séances d'acupuncture ne sont pas mentionnées comme remboursées pendant un traitement contre le cancer, et les pratiques de soins non conventionnelles comme l'acupuncture peuvent soulager mais ne remplacent pas les traitements habituels du cancer (DOC008_P032).",Information insuffisante dans les documents fournis.


In [7]:
# TEST (configuration figée) et question libre : toutes les réponses
duo[duo.expérience.str.startswith(("TEST", "LIBRE"))].assign(question=lambda d: d.question_id.map(
    lambda i: questions[i]["question"] if i in questions else QUESTION_LIBRE)).set_index(["question_id", "expérience"]).sort_index()[
    ["question", "OpenAI", "Mistral"]]

question  \
question_id expérience                                                                                                                                                                         
LIBRE       LIBRE Q1 v1 c2                                                                       Dans le cancer du sein, dans quel cas doit-on avoir une chimio adjuvante, ou néoadjuvante ?   
            LIBRE Q2 v1 c2                                                                       Dans le cancer du sein, dans quel cas doit-on avoir une chimio adjuvante, ou néoadjuvante ?   
Q021        TEST Q1 v1 c2                                  Pour le casque réfrigérant, combien de temps avant la perfusion est-il posé, à quel rythme est-il changé et quand l'enlève-t-on ?   
            TEST Q2 v1 c2                                  Pour le casque réfrigérant, combien de temps avant la perfusion est-il posé, à quel rythme est-il changé et quand l'enlève-t-on ?   
            TEST Q2 v1 c2, dense                           Pour le casque réfrigérant, combien de temps avant la perfusion est-il posé, à quel rythme est-il changé et quand l'enlève-t-on ?   
Q022        TEST Q1 v1 c2                                                                  Les consultations chez un psychologue sont-elles remboursées quand on est soigné pour un cancer ?   
            TEST Q2 v1 c2                                                                  Les consultations chez un psychologue sont-elles remboursées quand on est soigné pour un cancer ?   
            TEST Q2 v1 c2, dense                                                           Les consultations chez un psychologue sont-elles remboursées quand on est soigné pour un cancer ?   
Q023        TEST Q1 v1 c2                         Sous capécitabine, j'ai les paumes et les plantes de pieds rouges et douloureuses : de quoi s'agit-il et comment prendre soin de ma peau ?   
            TEST Q2 v1 c2                         Sous capécitabine, j'ai les paumes et les plantes de pieds rouges et douloureuses : de quoi s'agit-il et comment prendre soin de ma peau ?   
            TEST Q2 v1 c2, dense                  Sous capécitabine, j'ai les paumes et les plantes de pieds rouges et douloureuses : de quoi s'agit-il et comment prendre soin de ma peau ?   
Q024        TEST Q1 v1 c2                                                                       Je suis en chimiothérapie : est-ce que je peux faire mon vaccin contre la grippe cet hiver ?   
            TEST Q2 v1 c2                                                                       Je suis en chimiothérapie : est-ce que je peux faire mon vaccin contre la grippe cet hiver ?   
            TEST Q2 v1 c2, dense                                                                Je suis en chimiothérapie : est-ce que je peux faire mon vaccin contre la grippe cet hiver ?   
Q025        TEST Q1 v1 c2                                       J'ai envie de vomir dès que j'arrive à l'hôpital, avant même que la perfusion commence : est-ce connu et que peut-on faire ?   
            TEST Q2 v1 c2                                       J'ai envie de vomir dès que j'arrive à l'hôpital, avant même que la perfusion commence : est-ce connu et que peut-on faire ?   
            TEST Q2 v1 c2, dense                                J'ai envie de vomir dès que j'arrive à l'hôpital, avant même que la perfusion commence : est-ce connu et que peut-on faire ?   
Q026        TEST Q1 v1 c2         Le médecin veut me prescrire des compléments nutritionnels oraux : sous quelle forme existent-ils, sont-ils remboursés et peuvent-ils remplacer un repas ?   
            TEST Q2 v1 c2         Le médecin veut me prescrire des compléments nutritionnels oraux : sous quelle forme existent-ils, sont-ils remboursés et peuvent-ils remplacer un repas ?   
            TEST Q2 v1 c2, dense  Le médecin veut me prescrire des compléments nutritionnels oraux : sous quelle forme existent-ils, sont-ils rembou

## Lecture

**Chiffres (contrôles automatiques et BLEU/ROUGE)**
- Avec les mêmes prompts, Mistral obtient un BLEU et un ROUGE un peu meilleurs dans presque toutes les expériences
  (configuration figée, TEST : Q2 ROUGE-L 0,359 contre 0,325 ; Q2 dense 0,380 contre 0,354). Ses réponses sont plus
  courtes (médiane 40 mots contre 59 pour Q2 sur TEST), ce qui favorise le BLEU. Ces écarts restent petits sur 8 à
  16 questions et ne mesurent pas la justesse.
- **Abstentions** : sur TEST, Mistral Q2 s'abstient correctement sur les deux questions sans réponse (OpenAI : 1 sur 2,
  Q030 répondu sans la phrase imposée). Sur DEV, avec les passages BM25, il ne s'abstient à tort que sur Q016 (preuve
  absente) ; OpenAI ajoutait aussi une abstention partielle inutile sur Q014. Q1 ne s'abstient jamais, avec les deux LLM.
- **Citations** : aucune citation inventée chez les deux. Mistral cite plus souvent après chaque phrase, et met en gras
  les informations clés (mise en forme Markdown).
- Mistral répond un peu plus vite (≈ 0,9 s contre 1,1 à 1,4 s par réponse).

**Lecture des réponses (mêmes passages)**
- Les conclusions des notebooks 4.2 et 5 **ne changent pas** : la plupart des erreurs viennent de la recherche
  (Q012, Q016, Q025 avec BM25), et les deux LLM se trompent aux mêmes endroits. Q025 avec BM25 : aucun ne parle de
  nausées anticipatoires, mais Mistral ajoute « l'anxiété » et un « conditionnement psychologique », absents du
  passage DOC001_P087 (vrai médicalement, mais non soutenu). Avec la recherche dense, les deux répondent juste.
- **Q022 (sources divergentes)** : comme OpenAI, Mistral ne signale pas la divergence entre les deux documents.
- **Q024 (grippe)** : Mistral Q2 reste fidèle au texte (« la plupart des vaccinations »), alors qu'OpenAI Q2 généralisait
  au vaccin contre la grippe ; Mistral Q1 fait la même généralisation. **Q027** : Mistral n'ajoute pas le calcul
  « 2 h 30 par semaine » d'OpenAI Q2.
- **Q030 (acupuncture, sans réponse)** : avec Q1, OpenAI affirme à tort que ce n'est pas remboursé ; Mistral dit que
  les passages n'en parlent pas.
- **Essai 3 / 5 / 10 passages** : mêmes tendances qu'avec OpenAI (k = 3 fait perdre des preuves : 4 abstentions à
  tort). Avec 10 passages, Mistral s'abstient correctement sur Q016, là où OpenAI donnait une réponse non soutenue.
- **Question libre** : réponses équivalentes, justes et citées ; comme OpenAI, Mistral ne signale pas que les
  documents ne parlent pas du cancer du sein en particulier.
- **Reproductibilité** : malgré la température 0 et la graine fixe, 5 prompts TEST renvoyés une seconde fois sans
  cache donnent 3 réponses identiques sur 5 (vérification faite hors notebook) : c'est le cache qui rend les sorties
  stables, comme pour OpenAI.